# Aula 03 — capacidade, custos e decisão

Dados **simulados** com semente fixa. O escore é fornecido como insumo: ele não foi estimado por um modelo. O objetivo é comparar regras de priorização sob capacidade limitada e custos assimétricos.

In [ ]:
import numpy as np
import pandas as pd

SEMENTE = 20260917
rng = np.random.default_rng(SEMENTE)
n = 40
necessidade = rng.normal(size=n)
prob_real = 1 / (1 + np.exp(-(0.9 * necessidade - 0.4)))
dados = pd.DataFrame({
    'caso': np.arange(1, n + 1),
    'grupo': rng.choice(['A', 'B'], size=n, p=[0.55, 0.45]),
    'escore': np.clip(prob_real + rng.normal(0, 0.16, n), 0.01, 0.99),
    'baseline_operacional': necessidade + rng.normal(0, 0.8, n),
    'y': rng.binomial(1, prob_real),
})
dados.head()

## Métricas da fila
Para os $k$ casos priorizados, calcularemos $TP_k$, $FP_k$, precision@$k$ e $V_k=B\,TP_k-C\,FP_k$.

In [ ]:
def avaliar_fila(df, coluna_ordem, k, B=4, C=1, crescente=False):
    topo = df.sort_values(coluna_ordem, ascending=crescente).head(k)
    tp = int(topo['y'].sum())
    fp = int(k - tp)
    return {
        'regra': coluna_ordem, 'k': k, 'TP': tp, 'FP': fp,
        'precision@k': tp / k, 'beneficio_liquido': B * tp - C * fp
    }

k = 10
resultado = pd.DataFrame([
    avaliar_fila(dados, 'escore', k),
    avaliar_fila(dados, 'baseline_operacional', k),
])
resultado

In [ ]:
# Referência aleatória: média de 2.000 filas, preservando a capacidade k.
beneficios_aleatorios = []
for _ in range(2000):
    topo = dados.sample(k, random_state=int(rng.integers(0, 2**32 - 1)))
    tp = int(topo['y'].sum())
    beneficios_aleatorios.append(4 * tp - 1 * (k - tp))

pd.Series(beneficios_aleatorios).describe(percentiles=[0.05, 0.5, 0.95])

## Curva de capacidade
Observe como a escolha de $k$ muda precisão e benefício. Mais ações podem gerar mais verdadeiros positivos e também consumir recursos em falsos positivos.

In [ ]:
curva = pd.DataFrame([
    avaliar_fila(dados, 'escore', capacidade, B=4, C=1)
    for capacidade in range(2, 21, 2)
])
curva

In [ ]:
try:
    import plotly.express as px
    px.line(curva, x='k', y=['precision@k', 'beneficio_liquido'], markers=True,
            title='Desempenho da fila por capacidade (dados simulados)')
except ImportError:
    print('Plotly não instalado; a tabela acima contém todos os resultados.')

## Sensibilidade ao custo
Compare o resultado quando agir sobre um caso que não se beneficia fica duas vezes mais caro.

In [ ]:
sensibilidade = pd.DataFrame([
    avaliar_fila(dados, 'escore', capacidade, B=4, C=custo)
    | {'C': custo}
    for custo in [1, 2]
    for capacidade in [5, 10, 15]
])
sensibilidade[['C', 'k', 'TP', 'FP', 'precision@k', 'beneficio_liquido']]

## Entrega da dupla
1. Escolha uma capacidade $k$ e justifique-a.  
2. Compare fila por escore, baseline operacional e referência aleatória.  
3. Explique o que muda quando $C$ dobra.  
4. Escreva uma recomendação de três frases e uma limitação.  
5. Não apresente estes números como achados reais: a base é simulada.